# <u>INTRODUCTION</u>

The objective is to value/price European and Binary options using Monte Carlo simulation. We will compare the results between Euler-Maruyama scheme (EM), Milstein scheme and Closed Form. Through my findings we aim to identify which approach produces the best result that will give the most accurate and computationally efficient valuation for the option.

The stochastic differential equation (SDE) used to model the underlying asset price is the Geometric Brownian Motion (GBM):

$$
dS_t = r S_t\, dt + \sigma S_t\, dX_t
$$

where

$$r \text{ is the risk-free rate}$$
$$\sigma \text{ is the volatility}$$
$$X_t \text{ is a standard Brownian motion under the risk-neutral measure}$$

Each scheme aims to approximate the solution to this SDE through Monte Carlo simulation.

## Schemes

#### Closed-Form 

The known solution which allows us to simulate the terminal stock price exactly without discretization error:

$$
S(T) = S_0 \cdot \exp \left[ \left(r - \frac{1}{2} \sigma^2\right) T + \sigma \sqrt{T} \cdot \phi \right], \quad \phi \sim \mathcal{N}(0, 1)
$$

This method is considered exact for Monte Carlo simulations as it avoids time-stepping altogether.


#### Euler Maruyama

The **EM** method is a time-discretization approach for simulating the SDE:

$$
S_{n+1} = S_n + r S_n \delta t + \sigma S_n \sqrt{\delta t} \cdot \phi_n
$$

where  
$$
\phi_n \sim \mathcal{N}(0, 1), \quad \delta t = \frac{T}{N}
$$





#### Milstein

The **Milstein** scheme improves on EM by including an extra correction term from a stochastic Taylor expansion:

$$
S_{n+1} = S_n + r S_n \, \delta t + \sigma S_n \sqrt{\delta t} \cdot \phi_n + \frac{1}{2} \sigma^2 S_n \left( \phi_n^2 - 1 \right) \delta t
$$

where  
$$
\phi_n \sim \mathcal{N}(0, 1), \quad \delta t = \frac{T}{N}
$$


## Methodology
Throughout this report, we consider European and Binary call options. The methods and analysis presented would be equally applicable to put options with the appropriate change in payoff function.

#### <u> Payoffs</u>

The payoff we will use is:

- European call:  
  $$
  \max(S_T - E,\, 0)
  $$

- Binary (cash-or-nothing) call:  
  $$
  \mathbf{1}_{\{S_T > E\}}
  $$
   Pays 1 unit of cash if in-the-money

#### <u> Assumptions</u>
Risk Neutral world since we are using expected payoff under the risk neutral density.



#### <u> Estimators</u>

For $M$ paths and discount $D = e^{-rT}$:

$$
\hat V \;=\; D \cdot \frac{1}{M}\sum_{m=1}^{M} f\!\left(\hat S_T^{(m)}\right).
$$

Standard error (SE):
$$
\mathrm{SE} \;=\; D \,\frac{\mathrm{sd}\!\left(f\!\left(\hat S_T\right)\right)}{\sqrt{M}}.
$$
Report $95\%$ CI as $\hat V \pm 1.96\,\mathrm{SE}$.

For Bias $= \hat V - C_{BS}$

We use Black–Scholes benchmark:
$$
C_{BS} \;=\; S_0 N(d_1) - E e^{-rT} N(d_2), \qquad
C^{\mathrm{binary}}_{BS} \;=\; e^{-rT} N(d_2).
$$

With

$$
d_1=\frac{\ln(S_0/E)+(r+\tfrac{1}{2}\sigma^2)T}{\sigma\sqrt{T}}, \qquad
d_2=d_1-\sigma\sqrt{T}.
$$


And
$$
C_{\text{binary}}
= e^{-rT}\,\mathbb{E}^{\mathbb{Q}}\!\big[Q \cdot \mathbf{1}_{\{S_T>E\}}\big]
= Q\,e^{-rT}\,\mathbb{Q}(S_T>E)
= Q\,e^{-rT}\,N(d_2)
= e^{-rT}\,N(d_2).
$$

*(last step uses \(Q=1\), i.e., unit cash payoff)*


#### <u>Parameter Sweeps</u>
The current base parameter is:

$$
\begin{aligned}
\text{Today's stock price } S_0 & = 100 \\
\text{Strike } E & = 100 \\
\text{Time to expiry } (T - t) & = \text{1 year} \\
\text{volatility } \sigma & = 20\% \\
\text{constant risk-free interest rate } r & = 5\%
\end{aligned}
$$


A parameter sensitivity analysis was performed to evaluate model behavior across different simulation and allocation assumptions. The final configuration was selected based on a balance of performance, stability, drawdown behavior, and implementation realism rather than maximizing a single metric.

Detailed sweep tables and exam-specific parameter outputs have been omitted from this public version.

#### <u>Convergence: Strong vs Weak </u>

From Higham (2001) and Kloeden & Platen (1992) , we know that the convergence results are written as inequalities with a constant $C$

$$
\Big(\mathbb{E}\,|S_T^{(N)}-S_T|^p\Big)^{1/p} \;\le\; C\,(\delta t)^{\alpha}
$$

and for weak error,

$$
\big|\mathbb{E}[f(S_T^{(N)})]-\mathbb{E}[f(S_T)]\big| \;\le\; C\,(\delta t)^{\beta}
$$

Here $C$ is a constant independent of $\delta t$ (and typically of $N=T/\delta t$), though it may depend on $T$, moments of the solution, and growth constants of $\mu, \sigma$.

If there exists a $\delta t_{0}>0$ and a constant $C<\infty$ such that for all $0<\delta t\le \delta t_{0}$ then for

$$
\mathrm{error}(\delta t) \;\le\; C\,(\delta t)^{\alpha},
$$

we write

$$
\mathrm{error}(\delta t) \;=\; \mathcal{O}\big((\delta t)^{\alpha}\big) \quad \text{as } \delta t \to 0.
$$


**Strong convergence** measures **pathwise accuracy** of the terminal value (or the whole path):

$$
\Big(\mathbb{E}\,|S_T^{(N)}-S_T|^p\Big)^{1/p}
= \mathcal{O}\!\big((\delta t)^{\alpha}\big)
$$

EM has $$\alpha=\tfrac{1}{2}$$ Milstein has $$\alpha=1$$

**Weak convergence** measures **accuracy of expectations** (what option pricing needs):

$$
\big|\mathbb{E}[f(S_T^{(N)})]-\mathbb{E}[f(S_T)]\big|
= \mathcal{O}\!\big((\delta t)^{\beta}\big)
$$

For smooth payoffs, both EM and Milstein typically have $$\beta=1$$



#### Theoretical Summary:

**EM Error behavior**:

- Strong convergence order:  
  $
  \mathcal{O}(\delta t^{1/2})
  $

- Weak convergence order:  
  $
  \mathcal{O}(\delta t)
  $

**Milstein Error behavior**:

- Strong convergence order:  
  $
  \mathcal{O}(\delta t)
  $

- Weak convergence order:  
  $
  \mathcal{O}(\delta t)
  $

#### <u>Weak convergence test (weak order check):</u>

We empirically assess the **weak order of convergence** $\beta$ by assuming the bias decays like

$$
|{\rm bias}_N|
\;=\;
\big|\hat V_N - V_{\text{BS}}\big|
\;\approx\;
C\,(\delta t)^{\beta},
\qquad
\delta t = \tfrac{T}{N}.
$$

Taking logs, we fit
$$
\log |{\rm bias}_N|
\;=\;
\log C \;+\; \beta\,\log(\delta t).
$$

Using ordinary least squares (excluding noise-dominated points), the estimated slope $\hat{\beta}$ reveals the empirical weak order. This mirrors the test of Altmayer & Neuenkirch (2016) in their analysis of weak convergence. 

For smooth payoffs (e.g., European call) we expect $\beta \approx 1$; for discontinuous payoffs (e.g., binary call) $\beta \approx \tfrac12$ due to degradation in weak convergence near the discontinuity.


For EM and Milstein, run $N \in \{4,8,16,32,64,128,256\}$ with fixed $M=200{,}000$.  

and we plot $|\mathrm{bias}|$ vs $N$ on log–log axes; estimate slope.

As we are focused on expectation of price rather than pathwise error, we will not do a check on strong order.

We will apply this just for the base case


#### <u> Bias Constant C test</u>

From the fitted model $|{\rm bias}_N| \approx C\,(\delta t)^{\beta}$, we extract the **bias constant** $C$ as:

$$
\hat C = \exp\!\big(\text{intercept from log–log regression}\big).
$$

Alternatively, with theoretical $\beta$ fixed ( $1$ for European, $\tfrac12$ for binary), we compute
$$
\log C \;=\; \operatorname{mean}\!\big[\log |{\rm bias}_N| \;-\; \beta \log(\delta t)\big].
$$

Comparing $C_{\mathrm{EM}}$ vs $C_{\mathrm{Milstein}}$ quantifies the practical difference in bias magnitude between schemes at the same order. A **smaller** $C$ reflects **greater efficiency**.
    
**Why this is needed**
    
Upon reading Giles (2006), it emphasizes the importance of bias reduction to improve multilevel convergence. Though not directly implemented form the article, we empirically estimate \(C\) by fitting a log–log regression of the observed bias across different step sizes.

Weak convergence test tells us how fast the bias decreases with smaller timesteps (via $\beta$), but it does not tell us how big the bias is at practical step sizes.     

This provides a complementary insight to the convergence rate $\beta$ by quantifying how large the bias actually is for a given time step $\delta t$, even if the decay rate is acceptable.
    
We will apply this just for the base case.

#### <u> Bias - SE Crossover Cost Test</u>

To understand the practical trade-off between time discretization and Monte Carlo sampling error, we plot both **bias** and **standard error (SE)** against $\delta t$. The **crossover point**—where $|{\rm bias}_N|\approx \mathrm{SE}_N$—marks the point beyond which further time refinement does not reduce $\mathrm{RMSE}$ unless the number of Monte Carlo paths $M$ is increased. This helps identify the “efficiency ceiling” of time refinement under a fixed computational budget.
    
This test is an attempt to conceptually match adaptive sampling done in Giles (2022). The logic of adaptively adjusting the number of samples per time step is built on the principle of balancing discretization bias with standard error.
    
We will apply this just for the base case

#### <u> Accuracy Cost Test</u>

Following Harwell (2019), Giles (2008) and Glasserman(2003), we plot Root Mean Squared Error (RMSE) versus computational cost such as runtime or number of simulations to compare which scheme is most efficient at achieving a given level of accuracy. This is known as the cost-accuracy frontier.

 **Cost–accuracy frontier:** record wall-clock runtime for each $(\text{scheme},N)$.  
  Plot
  $
  \mathrm{RMSE} \;\approx\; \sqrt{\text{bias}^2 + \mathrm{SE}^2}
  $
  vs runtime; the better method lies lower-left.
  
  We will just focus just on the base case.

### Additional Data Filter

#### <u> Variance reduction</u>

In order optimize our Monte Carlo Simulation we use both Common Random Numbers (CRN) from Robert & Casella (2004) and Antithetic Variates from Glasserman (2003) to apply variance reduction techniques.

**Common Random Numbers (CRN):** By reusing the same stream of random numbers across all three schemes and across all time discretizations \(N\), we ensure a fair and consistent comparison. This reduces simulation noise in differences between schemes, isolating their true performance differences.

**Antithetic variates:** or each simulation path using a random normal variate \(Z\), we also simulate a mirror path using \(-Z\). Averaging the two payoffs reduces the variance of the estimator without affecting the bias. This leverages symmetry in Brownian increments to cancel randomness more efficiently.

#### <u> K noise</u>

To ensure stable and meaningful estimates of the bias constant $C$ and the bias–SE crossover point, I apply a **noise filter** to minimize error in analysis, a principle consistent with Zwicker et al. (2015). We use a user-defined threshold $k_{\text{noise}}$, specifically, we exclude time steps where the absolute bias falls below $k_{\text{noise}}\!\times\!\mathrm{SE}$; equivalently, I keep points satisfying
$$
\lvert \mathrm{bias}\rvert \;>\; k_{\text{noise}}\cdot \mathrm{SE}. 
$$

This filter is **only** applied in three contexts:
1. **Weak Convergnce test**, to keep weak order fit such that it exludes noise dominated points
2. **Bias constant estimation** (fixed-slope regression with $\beta=1$), to avoid distortion from statistically insignificant bias values.  
3. **Bias–SE crossover detection**, to prevent false crossover identification due to stochastic noise.

I set $k_{\text{noise}}=1.5$ in our experiments, which effectively discards low-signal points without overly reducing the dataset.

---

## Hypothesis

1. For GBM under $\mathbb{Q}$, the closed-form scheme will match the Black–Scholes benchmarks within the sampling error. 
2. Between time-stepping schemes, **Milstein** will exhibit smaller bias than **EM** at fixed $N$, reflecting strong order $1$ vs $\tfrac{1}{2}$ for EM. 
3. For **European (smooth) payoffs**, both EM and Milstein will show weak order $\approx 1$
4. For **Binary (discontinuous) payoffs**, weak order will degrade (often $\approx \tfrac{1}{2}$), so convergence will be visibly slower, especially near-ATM.

---

# <u> RESULTS</u>

## <u>Base Case</u>

$$
\begin{aligned}
\text{Today's stock price } S_0 & = 100 \\
\text{Strike } E & = 100 \\
\text{Time to expiry } (T - t) & = \text{1 year} \\
\text{volatility } \sigma & = 20\% \\
\text{constant risk-free interest rate } r & = 5\%
\end{aligned}
$$

In this section we create function definitions and check that the code runs well. In the next section, the summary table will include all cases and uses common random numbers (CRN) & Antithetic variates across all methods to allow for low-noise comparisons.

### Base Case: Closed Form European

In [ ]:
# Redacted

### Base Case: EM European

In [ ]:
# Redacted

### Base Case: Milstein European

In [ ]:
# Redacted

### Base Case: Close Form Binary

In [ ]:
# Redacted

### Base Case: EM Binary

In [ ]:
# Redacted

### Base Case: Milstein Binary

In [ ]:
# Redacted

## <u> All Cases: European and Binary (Raw Results) </u>

In [ ]:
# Redacted

## <u> Weak Convergence Diagnostic: Base Case</u>

In [ ]:
# Redacted

In [ ]:
# Redacted

## <u>Bias Constant C</u>

In [ ]:
# Redacted

In [ ]:
# Redacted

## <u> Bias-SE Cross over: Base Case</u>

In [ ]:
# Redacted

In [ ]:
# Redacted

## <u> Accuracy Cost Test - Base Case</u>

In [ ]:
# Redacted

In [ ]:
# Redacted

In [ ]:
# Redacted

# <u> Observations </u>

### <u> Raw Results Breakdown</u>

## Observations

### Raw Results Breakdown

Across the tested configurations, the closed-form Black–Scholes value was used as the benchmark for evaluating the numerical schemes. The public version omits the full case-by-case results, exact errors, and exam-specific parameter comparisons.

### European Options

For European call options, Euler–Maruyama and Milstein both produced results consistent with the theoretical benchmark. Milstein can benefit from its diffusion correction term, particularly when pathwise accuracy is relevant, but the practical advantage depends on the payoff, timestep size, volatility regime, and simulation setup.

The case-by-case numerical comparison has been omitted from this public version.

### Binary Options

For binary call options, the discontinuous payoff made convergence behavior less stable and reduced the practical benefit of higher-order pathwise correction. This highlights an important implementation point: a theoretically stronger discretization scheme does not always produce a materially better pricing result for nonsmooth payoffs.

The case-by-case numerical comparison has been omitted from this public version.

### Observation Pricing Summary

Overall, the exercise reinforced that numerical method selection should be based on validation against benchmarks, payoff structure, discretization error, Monte Carlo standard error, and computational cost rather than theoretical convergence order alone.

Detailed parameter cases, exact pricing errors, and exam-specific outputs have been redacted.

Note: Across the retained simulations, paths remained numerically stable, with no negative asset values observed during simulation or at maturity.

---

### <u> Weak Convergence Test: Base Case</u>

#### European Option

For the European call option, both Euler–Maruyama and Milstein were evaluated against the Black–Scholes benchmark using weak convergence diagnostics.

The public version omits fitted slopes, regression values, bias constants, retained point counts, and exact numerical outputs. Qualitatively, both schemes behaved broadly in line with theoretical expectations for smooth payoffs, though practical performance depended on finite-sample noise, timestep choice, and Monte Carlo standard error.

#### Binary Option

For the binary call option, convergence behavior was less stable because the payoff is discontinuous. This made the weak convergence diagnostic more sensitive to noise and reduced the practical benefit of higher-order pathwise correction.

The public version omits fitted slopes, regression values, bias constants, retained point counts, and exact numerical outputs.

#### Summary

This diagnostic reinforced that theoretical convergence order is only one part of numerical method selection. In practice, payoff smoothness, discretization bias, Monte Carlo standard error, and computational cost all affect whether a more sophisticated scheme provides a meaningful advantage.

### <u> Constant Bias C test</u>

#### European Option

For the European call option, the bias constant test was used to separate convergence rate from practical error magnitude. Both Euler–Maruyama and Milstein were evaluated against the benchmark to assess whether lower discretization bias translated into materially better pricing performance.

The public version omits fitted convergence rates, bias constants, regression values, and exact numerical outputs.

#### Binary Option

For the binary call option, the discontinuous payoff made the fitted bias relationship less stable. This reinforces that higher-order pathwise schemes do not automatically improve practical pricing accuracy for nonsmooth payoffs.

The public version omits fitted convergence rates, bias constants, regression values, and exact numerical outputs.

#### Summary

The test highlights that method selection should not rely only on theoretical convergence order. Practical pricing accuracy depends on the interaction between payoff smoothness, timestep size, discretization bias, Monte Carlo noise, and computational cost.

### <u> Bias-SE Crossover Test</u>

#### European Options

For European options, the crossover analysis was used to compare when discretization bias becomes small relative to Monte Carlo standard error. The results suggested that the higher-order scheme may reduce bias earlier in some settings, but this benefit must be weighed against implementation complexity, runtime, and initial bias behavior.

#### Binary Options

For binary options, the discontinuous payoff made the crossover behavior less stable. The analysis suggested that higher-order pathwise correction does not necessarily translate into better practical pricing accuracy for nonsmooth payoffs.

Overall, the test reinforced that numerical method selection should consider both bias decay and Monte Carlo uncertainty rather than convergence order alone.

### <u> Cost Accuracy Test</u>

#### European Option

**Closed-form benchmark:**  
The closed-form Black–Scholes value was used as the reference benchmark. It is effectively instantaneous and provides a useful anchor for evaluating simulation-based pricing error.

**Euler–Maruyama vs. Milstein:**  
Both discretization schemes were evaluated in terms of pricing error, Monte Carlo standard error, and runtime. In the tested setup, Milstein generally reduced discretization bias for smooth European payoffs, but this came with additional computational cost and did not always translate into a large practical improvement after accounting for Monte Carlo noise.

**Observation:**  
For European options, the higher-order scheme can be useful, but the practical benefit depends on sample size, timestep choice, runtime budget, and the size of the remaining Monte Carlo standard error.

#### Binary Option

**Closed-form benchmark:**  
The closed-form binary option value was used as the benchmark for assessing simulation accuracy.

**Euler–Maruyama vs. Milstein:**  
For discontinuous binary payoffs, the practical advantage of the higher-order scheme was less clear. The discontinuity in the payoff can make convergence and error behavior less stable, so lower theoretical pathwise error does not necessarily produce materially better pricing accuracy.

**Observation:**  
For nonsmooth payoffs, cost-accuracy tradeoffs should be evaluated empirically rather than assumed from convergence order alone. Detailed runtime figures and exact error comparisons have been omitted from this public version.

# <u> Analysis</u>

Detailed analysis has been redacted from this public version to avoid exposing exam-specific case results, fitted diagnostics, and exact numerical outputs.

The retained discussion focuses on the validation framework, numerical-method comparison, and qualitative implementation lessons.

# <u> Conclusion</u>

This project compared exact terminal GBM simulation, Euler–Maruyama, and Milstein discretization against closed-form Black–Scholes benchmarks for European and binary call options.

The public version omits exam-specific parameter cases, fitted convergence coefficients, exact pricing errors, crossover thresholds, and detailed cost-accuracy rankings.

### Key Takeaways

1. **Closed-form benchmarks are essential when available.**  
   Black–Scholes provided a useful reference point for evaluating simulation-based pricing methods and separating discretization error from Monte Carlo sampling error.

2. **Theoretical convergence order is not enough.**  
   A higher-order numerical scheme does not automatically produce better practical pricing performance once payoff structure, timestep choice, sampling noise, and runtime are considered.

3. **Payoff smoothness matters.**  
   European options behaved more consistently with standard convergence expectations, while binary options were more sensitive to payoff discontinuity and Monte Carlo noise.

4. **Method selection should be empirical and context-specific.**  
   Euler–Maruyama and Milstein should be compared using benchmark error, standard error, bias behavior, and computational cost rather than theoretical order alone.

5. **Implementation realism matters.**  
   The most robust pricing workflow is not necessarily the most complex one. Numerical method choice should be justified by practical accuracy improvement relative to its additional computational and implementation cost.

Overall, the project reinforced a validation-driven approach to numerical pricing: start with a benchmark, test error behavior empirically, account for Monte Carlo uncertainty, and avoid assuming that theoretical sophistication automatically improves practical results.

# <u>References

1. Glasserman, P. (2003). *Monte Carlo Methods in Financial Engineering*. Springer.

2. Higham, D. J. (2001). An algorithmic introduction to numerical simulation of stochastic differential equations. *SIAM Review, 43*(3), 525–546. [https://doi.org/10.1137/S0036144500378302](https://doi.org/10.1137/S0036144500378302)

3. Kloeden, P. E., & Platen, E. (1992). *Numerical Solution of Stochastic Differential Equations*. Springer.

4. Robert, C. P., & Casella, G. (2004). *Monte Carlo Statistical Methods* (2nd ed.). Springer.

5. Harwell, M. R. (2019). A strategy for using bias and RMSE as outcomes in Monte Carlo studies in statistics. *Journal of Modern Applied Statistical Methods, 17*(2), Article 5. [https://doi.org/10.22237/jmasm/1551907966](https://doi.org/10.22237/jmasm/1551907966)
 
6. Altmayer, M., & Neuenkirch, A. (2016). *Discretizing the Heston model: An analysis of the weak convergence rate*. arXiv preprint arXiv:1604.05540.  
https://arxiv.org/abs/1604.05540

7. Giles, M. B. (2006). *Improved multilevel Monte Carlo convergence using the Milstein scheme* (Tech. Rep. No. NA–06–22). University of Oxford.  
https://people.maths.ox.ac.uk/~gilesm/files/NA-06-22.pdf

8.  Giles, M. B. (2022, June). *MLMC techniques for discontinuous functions* [Conference presentation]. RWTH Aachen University.  
https://people.maths.ox.ac.uk/gilesm/talks/RWTH_22.pdf
    
9. Zwicker, M., Jarosz, W., Lehtinen, J., Moon, B., Ramamoorthi, R., Rousselle, F., Sen, P., Soler, C., & Yoon, S.-E. (2015). *Recent advances in adaptive sampling and reconstruction for Monte Carlo rendering*. Computer Graphics Forum, 34(2), 667–681. https://doi.org/10.1111/cgf.12592